In [3]:
from pathlib import Path

# Try the original path
SARTAJ_PATH = Path("/kaggle/input/datasets/sartajbhuvaji/brain-tumor-classification-mri")

if not SARTAJ_PATH.exists():
    print("❌ Path not found! Let's see what IS there:")
    # List everything in the input folder to find the real path
    import os
    for root, dirs, files in os.walk('/kaggle/input'):
        print(root)
else:
    print("✅ Path is correct! Ready to run.")

✅ Path is correct! Ready to run.


In [4]:
import os
import pandas as pd
import numpy as np
import json
import hashlib
from PIL import Image
from pathlib import Path
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from tqdm import tqdm
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# ==========================================
# 1. CONFIGURATION
# ==========================================
SARTAJ_PATH = Path("/kaggle/input/datasets/sartajbhuvaji/brain-tumor-classification-mri") 
HF_REPO_ID = "srijan2715/sartaj-clean-v3" # NEW REPO NAME
SEED = 42
label_to_int = {"glioma": 0, "meningioma": 1, "no": 2, "pituitary": 3}

# ==========================================
# 2. SCAN AND HASH
# ==========================================
print("🔍 SCANNING RAW FOLDERS...")
records = []
for split_name in ['Training', 'Testing']:
    split_path = SARTAJ_PATH / split_name
    if not split_path.exists(): continue
    for class_dir in split_path.iterdir():
        if not class_dir.is_dir(): continue
        class_name = class_dir.name.lower().replace("_tumor", "").replace("notumor", "no")
        for img_path in class_dir.glob("*.*"):
            if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                with open(img_path, 'rb') as f:
                    file_hash = hashlib.sha256(f.read()).hexdigest()
                records.append({
                    "split": "train" if split_name == "Training" else "test",
                    "image_path": str(img_path),
                    "label_str": class_name,
                    "hash": file_hash
                })

df = pd.DataFrame(records)
raw_count = len(df)
print(f"📊 TOTAL RAW IMAGES FOUND: {raw_count}")

# ==========================================
# 3. THE CRITICAL CLEANING STEP (WITH ASSERTIONS)
# ==========================================
print("\n🧹 CLEANING DATA...")
train_hashes = set(df[df['split']=='train']['hash'])
original_test_count = len(df[df['split']=='test'])

# Remove test images that exist in train
df_no_leak = df[~((df['split'] == 'test') & df['hash'].isin(train_hashes))]
print(f"   - After leakage removal: {len(df_no_leak)}")

# Remove internal duplicates
df_clean = df_no_leak.drop_duplicates(subset=['hash'])
print(f"   - After duplicate removal: {len(df_clean)}")

# CRITICAL ASSERTIONS: Crash if cleaning failed
assert len(df_clean) < raw_count, f"❌ CLEANING FAILED! Count did not drop. Raw: {raw_count}, Clean: {len(df_clean)}"
assert len(df_clean[df_clean['split']=='test']) < original_test_count, f"❌ LEAKAGE REMOVAL FAILED! Test count did not drop."

print("✅ SUCCESS: Dataset size reduced successfully.")

# ==========================================
# 4. SPLIT DATA
# ==========================================
train_df = df_clean[df_clean['split'] == 'train'].copy()
test_df = df_clean[df_clean['split'] == 'test'].copy()

train_split, val_split = train_test_split(
    train_df, test_size=0.15, stratify=train_df['label_str'], random_state=SEED
)
print(f"\n📐 FINAL SPLITS: Train={len(train_split)}, Val={len(val_split)}, Test={len(test_df)}")

# ==========================================
# 5. PREPROCESS (224x224)
# ==========================================
def process_images(df, split_name):
    images, labels_int = [], []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Processing {split_name}"):
        if row['label_str'] not in label_to_int: continue
        try:
            img = Image.open(row['image_path']).convert("RGB")
            img_final = img.resize((224, 224), Image.Resampling.BICUBIC)
            images.append(np.array(img_final))
            labels_int.append(label_to_int[row['label_str']])
        except: pass
    return images, labels_int

print("\n🖼️ RESIZING TO 224x224...")
train_imgs, train_lbls = process_images(train_split, "train")
val_imgs, val_lbls = process_images(val_split, "val")
test_imgs, test_lbls = process_images(test_df, "test")

# ==========================================
# 6. UPLOAD
# ==========================================
print("\n🚀 UPLOADING TO HUGGING FACE...")
dataset_dict = DatasetDict({
    "train": Dataset.from_dict({"image": train_imgs, "label": train_lbls}),
    "val": Dataset.from_dict({"image": val_imgs, "label": val_lbls}),
    "test": Dataset.from_dict({"image": test_imgs, "label": test_lbls})
})

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

dataset_dict.push_to_hub(HF_REPO_ID, token=hf_token, private=False, max_shard_size="500MB")
print(f"\n🎉 DONE! Link: https://huggingface.co/datasets/{HF_REPO_ID}")

🔍 SCANNING RAW FOLDERS...
📊 TOTAL RAW IMAGES FOUND: 3264

🧹 CLEANING DATA...
   - After leakage removal: 3176
   - After duplicate removal: 3027
✅ SUCCESS: Dataset size reduced successfully.

📐 FINAL SPLITS: Train=2360, Val=417, Test=250

🖼️ RESIZING TO 224x224...


Processing test: 100%|██████████| 250/250 [00:01<00:00, 172.30it/s]



🚀 UPLOADING TO HUGGING FACE...


Uploading the dataset shards:   0%|          | 0/2 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


🎉 DONE! Link: https://huggingface.co/datasets/srijan2715/sartaj-clean-v3


In [5]:
import numpy as np
import hashlib
from datasets import load_dataset

print(" Loading NEW Sartaj dataset (sartaj-clean-v3)...")
ds = load_dataset("srijan2715/sartaj-clean-v3", download_mode="force_redownload")

def get_image_hash(img_array):
    return hashlib.sha256(img_array.tobytes()).hexdigest()

train_hashes = set()
test_hashes = set()

print("🔍 Hashing Train split...")
for i in range(len(ds['train'])):
    h = get_image_hash(np.array(ds['train'][i]['image']))
    train_hashes.add(h)

print("🔍 Hashing Test split...")
for i in range(len(ds['test'])):
    h = get_image_hash(np.array(ds['test'][i]['image']))
    test_hashes.add(h)

overlap = len(train_hashes & test_hashes)

print("\n" + "="*60)
print("📊 FINAL RESULTS FOR SARTAJ-CLEAN-V3")
print("="*60)
print(f"Train: {len(ds['train'])} | Test: {len(ds['test'])}")
print(f"🚨 Train/Test Overlap: {overlap} images")
print("="*60)

if overlap == 0:
    print("🎉 SUCCESS! 100% CLEAN. NO DATA LEAKAGE!")
else:
    print("⚠️ Still leaking.")

 Loading NEW Sartaj dataset (sartaj-clean-v3)...


README.md:   0%|          | 0.00/536 [00:00<?, ?B/s]

data/train-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  755MB            

data/train-00000-of-00002.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  755MB            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2360 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/417 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/250 [00:00<?, ? examples/s]

🔍 Hashing Train split...
🔍 Hashing Test split...

📊 FINAL RESULTS FOR SARTAJ-CLEAN-V3
Train: 2360 | Test: 250
🚨 Train/Test Overlap: 127 images
⚠️ Still leaking.
